# News crawler

In [1]:
import requests
# %pip install feedparser
import feedparser
# %pip install googlenewsdecoder
from googlenewsdecoder import gnewsdecoder as gnd
import time


In [11]:
query_string = "https://news.google.com/rss/search?q=NVDA+before:2025-03-15+after:2025-03-08&hl=en-US&gl=US&ceid=US:en"
req = requests.get(query_string)
parsed_req = feedparser.parse(req.content)

In [4]:
parsed_req

{'bozo': False,
 'entries': [{'title': 'Is NVIDIA Corp. (NVDA) The Best Upside Stock To Buy Right Now? - Yahoo Finance',
   'title_detail': {'type': 'text/plain',
    'language': None,
    'base': '',
    'value': 'Is NVIDIA Corp. (NVDA) The Best Upside Stock To Buy Right Now? - Yahoo Finance'},
   'links': [{'rel': 'alternate',
     'type': 'text/html',
     'href': 'https://news.google.com/rss/articles/CBMif0FVX3lxTE84Y0gyazVBMXFKMmNXbXZCOEhUX0pmQ3VWUGlreUFkOXdRbTNubmVoaDhGdDR3anU2RkFBZjlMSkNWdUxVY1FXWXdLSFlaanNjZXR2UzBRa1UzdE5lUnUwSGkySm1yb3V1T3JENldyS2p1NU1udkpsQzRSZ3lLMzA?oc=5'}],
   'link': 'https://news.google.com/rss/articles/CBMif0FVX3lxTE84Y0gyazVBMXFKMmNXbXZCOEhUX0pmQ3VWUGlreUFkOXdRbTNubmVoaDhGdDR3anU2RkFBZjlMSkNWdUxVY1FXWXdLSFlaanNjZXR2UzBRa1UzdE5lUnUwSGkySm1yb3V1T3JENldyS2p1NU1udkpsQzRSZ3lLMzA?oc=5',
   'id': 'CBMif0FVX3lxTE84Y0gyazVBMXFKMmNXbXZCOEhUX0pmQ3VWUGlreUFkOXdRbTNubmVoaDhGdDR3anU2RkFBZjlMSkNWdUxVY1FXWXdLSFlaanNjZXR2UzBRa1UzdE5lUnUwSGkySm1yb3V1T3JENldyS2p1NU1udkpsQ

Using simple crawler for Google's RSS, it will not give the news links since Google encode them. To bypass this using `googlenewsdecoder` can *partially* solved this since at times it still can returns 492 error.

In [12]:
# https://news.google.com/rss/search?q=site:benzinga.com+NVIDIA+after:2025-01-01+before:2026-01-01&hl=en-US&gl=US&ceid=US:en
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def req_google_rss(gnews_link: str, retries: int = 2, delay:float = 5):
    for attept in range(retries):
        result = gnd(gnews_link)
        if result.get("status"):
            return result["decoded_url"]
        time.sleep(delay * (attept + 1))
    return None

In [6]:
encoded_link = parsed_req["entries"][0]["link"]
# try to fetch
encoded_req = requests.get(encoded_link, headers=HEADERS)
# decode the link
decoded_link = req_google_rss(encoded_link)
# try to fetch using decoded link
decoded_req = requests.get(decoded_link, headers=HEADERS)
#
print(f"Encoded link: {encoded_req}")
print(f"Dencoded link: {decoded_req}")

Encoded link: <Response [200]>
Dencoded link: <Response [200]>


## Get news content

In [2]:
#%pip install trafilatura
#%pip install --user trafilatura
import trafilatura

In [15]:
def get_article_full_text(url: str, headers: dict):
    res = requests.get(url, headers=headers, timeout=10)
    if res.status_code != 200:
        return None
    return trafilatura.extract(res.text, include_comments=False, include_tables=False)

In [ ]:
get_article_full_text(decoded_link, HEADERS)

'We recently published a list of 10 Best Upside Stocks To Buy Right Now. In this article, we are going to take a look at where NVIDIA Corp. (NASDAQ:NVDA) stands against other best upside stocks to buy right now.\nOn March 8, Bob Elliott, Co-Founder, CEO, and CIO of Unlimited, and Kara Murphy, CIO of Kestra Investment Management, joined \'Closing Bell Overtime\' on CNBC to talk about the week\'s market action. In a discussion on whether stocks or gold were the better choice in the current economic climate, Bob Elliott noted that stocks were facing tough circumstances due to elevated expectations at the start of the year, which had begun to adjust downward. He highlighted concerns about fiscal tightening, tariff volatility, and weaker employment conditions. However, he emphasized that these factors were overshadowed by potential tax policy changes, immigration restrictions, and efforts to curb federal spending, which could impact nominal GDP growth. Kara Murphy was asked about diversific

## Selecting news

In [4]:
from datetime import date, datetime
from urllib.parse import urlencode
from urllib.parse import urlparse

In [ ]:
def query_builder(company_name: str, ticker: str, start_date: date, end_date: date, sites: list[str]):
    query = f"({ticker} OR {company_name})"
    if len(sites) == 1:
        query += f" site:{sites[0]}"
    else:
        site_join = " OR ".join(f"site:{s}" for s in sites)
        query += f" ({site_join})"
    
    query += f" after:{start_date.isoformat()} before:{end_date.isoformat()}"
    params = {
        "q": query,
        "hl": "en-US",
        "gl": "US",
        "ceid": "US:en",
    }
    return f"https://news.google.com/rss/search?{urlencode(params)}"


In [52]:
FINANCE_PUBLISHERS = ["finance.yahoo.com", "benzinga.com"]

url = query_builder(
    ticker="NVDA",
    company_name="NVIDIA",
    start_date=date(2025, 3, 8),
    end_date=date(2025, 3, 15),
    sites=FINANCE_PUBLISHERS,
)
print(url)

https://news.google.com/rss/search?q=%28NVDA+OR+NVIDIA%29+%28site%3Afinance.yahoo.com+OR+site%3Abenzinga.com%29+after%3A2025-03-08+before%3A2025-03-15&hl=en-US&gl=US&ceid=US%3Aen


In [74]:
def news_filter(link: str, start_date: date, end_date: date, sites: list[str]):
    req = requests.get(link, headers=HEADERS)
    if req.status_code != 200:
        return None
    parsed_req = feedparser.parse(req.content)
    filtered_entries = []
    for entry in parsed_req.entries:
        # CHECK DATE RANGE
        pub_date = datetime(*entry["published_parsed"][:6]).date()
        entry["published_time_formated"] = pub_date.strftime("%Y-%m-%d")
        if not (start_date <= pub_date <= end_date):
            continue
        # CHECK SITES
        source_href = entry.get("source", {}).get("href", "")
        domain = urlparse(source_href).netloc.replace("www.", "")
        if not any(domain == site or domain.endswith(f".{site}") for site in sites):
            continue
        filtered_entries.append(entry)
    return filtered_entries

In [75]:
results = news_filter(
    link=url,
    start_date=date(2025, 3, 8),
    end_date=date(2025, 3, 11),
    sites=FINANCE_PUBLISHERS,
)

print(f"{len(results)} entries matched after filtering")

31 entries matched after filtering


It is recommended to use 2-3 sites since it can distrupt Google's RSS filter.

In [ ]:
for row in results:
    #
    datetime(*entry["published_parsed"][:6]).date()

Nvidia RTX Pro 6000 Blackwell GPU spotted with 24,064 CUDA cores, 96GB GDDR7, and 600W — 11% more cores than RTX 5090 - Yahoo Finance
Nvidia has gross margins above 70%, its server hardware partners would be lucky to break 7% and things won't get better - Yahoo Finance
Why Nvidia Stock Is Sinking Again Today - Yahoo Finance
Could Nvidia Stock Help You Retire a Millionaire? - Yahoo Finance
Exclusive-TSMC pitched Intel foundry JV to Nvidia, AMD and Broadcom, sources say - Yahoo Finance
Famous Fund Manager Explains Why He Sold NVIDIA (NVDA) Position After Earnings — ‘I Saw The Wind’ - Yahoo Finance
NVIDIA Loses $1 Trillion in Market Value, Stock Tumbles 20% YTD - Yahoo Finance
'Astonishing' $1.57tn wiped off Mag 7 since start of 2025 - uk.finance.yahoo.com
META_TITLE_QUOTE - Yahoo Finance
If You Invest $10K in the Magnificent 7 Now, What Could It Be Worth in 10 Years? - Yahoo Finance
Shein elevates online shopping experience with Trend Stores - Yahoo Finance
Is United Microelectronics Cor

In [80]:
results

[{'title': 'Nvidia RTX Pro 6000 Blackwell GPU spotted with 24,064 CUDA cores, 96GB GDDR7, and 600W — 11% more cores than RTX 5090 - Yahoo Finance',
  'title_detail': {'type': 'text/plain',
   'language': None,
   'base': '',
   'value': 'Nvidia RTX Pro 6000 Blackwell GPU spotted with 24,064 CUDA cores, 96GB GDDR7, and 600W — 11% more cores than RTX 5090 - Yahoo Finance'},
  'links': [{'rel': 'alternate',
    'type': 'text/html',
    'href': 'https://news.google.com/rss/articles/CBMigAFBVV95cUxPWkJXdFR6YndBYi1jeGlDUHA2ZTBObnZrdlBNR3J4NGc4UVFEcURxSXJjUWxTQjgzZUNISDNVVDBMdlE2RWpKUzVhcEctZHNyeE9PSWdfR1F4WENhM2I2OUFqdDRMQnVFZnNOSDEtR3pyX1hBZE1GQlB6R0lnMnhZRg?oc=5'}],
  'link': 'https://news.google.com/rss/articles/CBMigAFBVV95cUxPWkJXdFR6YndBYi1jeGlDUHA2ZTBObnZrdlBNR3J4NGc4UVFEcURxSXJjUWxTQjgzZUNISDNVVDBMdlE2RWpKUzVhcEctZHNyeE9PSWdfR1F4WENhM2I2OUFqdDRMQnVFZnNOSDEtR3pyX1hBZE1GQlB6R0lnMnhZRg?oc=5',
  'id': 'CBMigAFBVV95cUxPWkJXdFR6YndBYi1jeGlDUHA2ZTBObnZrdlBNR3J4NGc4UVFEcURxSXJjUWxTQjgzZUNISD